In [ ]:
!pip uninstall -y torchvision
!pip install -q transformers datasets accelerate sentencepiece scikit-learn huggingface_hub

In [ ]:
import torch
import transformers
import datasets

print(torch.__version__)
print(transformers.__version__)
print(datasets.__version__)

In [ ]:
import pandas as pd
import numpy as np
import torch
import os

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
from google.colab import files
import io

uploaded = files.upload()  # 여기서 emotion_data.csv 선택
filename = next(iter(uploaded))
print("업로드된 파일:", filename)


def _read_csv_flexible(data: bytes) -> pd.DataFrame:
    for encoding in ("utf-8", "utf-8-sig", "cp949"):
        try:
            return pd.read_csv(io.BytesIO(data), encoding=encoding)
        except UnicodeDecodeError:
            continue
    raise ValueError("CSV 인코딩을 확인해주세요 (utf-8/utf-8-sig/cp949 모두 실패)")


df = _read_csv_flexible(uploaded[filename])
print(df.head())
print(df.columns)
print(df.shape)

In [ ]:
df = df[["사람문장1", "감정_대분류"]]

df = df.rename(
    columns={
        "사람문장1": "text",
        "감정_대분류": "emotion"
    }
)

df = df.dropna()

print(df.head())
print(df.shape)

In [ ]:
label_map = {
    "불안": 0,
    "분노": 1,
    "상처": 2,
    "슬픔": 3,
    "당황": 4,
    "기쁨": 5
}

id_to_label = {v: k for k, v in label_map.items()}

df = df[df["emotion"].isin(label_map.keys())].copy()
df["label"] = df["emotion"].map(label_map).astype(int)

print(df["emotion"].value_counts())
print(df["label"].value_counts())
print(df.shape)

In [ ]:
df = df.groupby("label", group_keys=False).apply(
    lambda x: x.sample(min(len(x), 2000), random_state=42)
).reset_index(drop=True)

print(df["emotion"].value_counts())
print(df["label"].value_counts())
print(df.shape)

In [ ]:
train_df, valid_df = train_test_split(
    df[["text", "label"]],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
valid_dataset = Dataset.from_pandas(valid_df, preserve_index=False)

print(train_dataset)
print(valid_dataset)

In [ ]:
model_name = "klue/bert-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=6
)

print(model.config.num_labels)

In [ ]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
valid_dataset = valid_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column("label", "labels")
valid_dataset = valid_dataset.rename_column("label", "labels")

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "token_type_ids", "labels"]
)

valid_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "token_type_ids", "labels"]
)

print(train_dataset[0])

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./bert_emotion_model",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
eval_result = trainer.evaluate()
print(eval_result)

In [ ]:
save_path = "./final_emotion_model"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("모델 저장 완료:", save_path)

In [ ]:
def predict_emotion(text: str) -> str:
    device = next(model.parameters()).device
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=64)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    pred_id = int(torch.argmax(logits, dim=1).item())
    return id_to_label[pred_id]

print(predict_emotion("저녁을 같이 먹기로 했는데 다들 바빠서 결국 혼자 먹었다."))
print(predict_emotion("오늘 좋은 소식을 들어서 기분이 너무 좋았다."))
print(predict_emotion("계속 같은 실수를 해서 너무 화가 났다."))

## 5. Hugging Face Hub에 업로드 (ai-server에서 쓸 수 있게)

`Backend/ai-server`는 `.env`의 `EMOTION_MODEL_PATH`로 지정된 경로에서 모델을 불러온다.
Colab 세션 안의 로컬 경로는 세션이 끝나면 사라지므로, Hugging Face Hub에 올리고
그 repo id를 `EMOTION_MODEL_PATH`에 넣어줘야 한다.

1. https://huggingface.co/settings/tokens 에서 Write 권한 토큰 발급
2. 아래 셀 실행 후 프롬프트에 토큰 붙여넣기
3. `repo_id`를 본인(or 팀) 계정 이름으로 바꾸기
4. 업로드 후 `Backend/ai-server/.env`에 다음처럼 반영:
   ```
   EMOTION_MODEL_PATH=repo_id에-적은-값
   HF_TOKEN=방금 만든 토큰  # repo를 private으로 올렸다면 필요
   ```

In [ ]:
from huggingface_hub import login

login()  # 토큰 입력 프롬프트가 뜬다

repo_id = "your-hf-username/mira-emotion-klue-bert"  # 실제 계정 이름으로 바꿀 것

model.push_to_hub(repo_id, private=True)
tokenizer.push_to_hub(repo_id, private=True)

print("업로드 완료:", repo_id)